In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load Letterboxd movies dataset
df = pd.read_csv('../src/letterboxd_movies_dataset.csv')

print(f"Letterboxd Dataset Loaded: {df.shape[0]} films")

# Load movies.csv and merge ONLY for matching films (memory efficient)
print("Loading additional movie metadata...")
df_movies = pd.read_csv('../movies.csv')[['id', 'name', 'description', 'rating']]

# Remove duplicates in movies.csv to prevent many-to-many join (keep first occurrence)
df_movies = df_movies.drop_duplicates(subset=['name'], keep='first')

# Load themes data
print("Loading themes information...")
themes_df = pd.read_csv('../themes.csv')
themes_grouped = themes_df.groupby('id')['theme'].apply(lambda x: ', '.join(x)).reset_index()
themes_grouped.columns = ['id', 'themes']

# Load actors and get top 3 cast per film
print("Loading cast information...")
actors_df = pd.read_csv('../actors.csv')
# Remove NaN values from actor names and get top 3 per film
cast_df = actors_df.dropna(subset=['name']).groupby('id')['name'].apply(lambda x: ', '.join(x.head(3))).reset_index()
cast_df.columns = ['id', 'cast']

# Merge cast and themes with movies
df_movies = df_movies.merge(cast_df, on='id', how='left')
df_movies = df_movies.merge(themes_grouped, on='id', how='left')

# Merge: only keep description, rating, cast, and themes for films that exist in Letterboxd
df = df.merge(df_movies, left_on='title', right_on='name', how='left', suffixes=('', '_movies'))

# Clean up: drop duplicate 'name' column and id column from movies.csv (keep 'title' from Letterboxd)
df = df.drop(columns=['name', 'id'], errors='ignore')

# Rename columns for consistency
df = df.rename(columns={
    'title': 'name',
    'year': 'date',
    'runtime': 'minute',
    'genres': 'genre'
})

# Clean up duplicates in genre column (remove repeated genres)
df['genre'] = df['genre'].str.replace(r'(\w+)(,\s*\1)+', r'\1', regex=True)

# Keep only essential columns for recommendation system
essential_cols = ['name', 'date', 'genre', 'primary_genre', 'minute', 'movie_era', 'description', 'rating', 'cast', 'themes']
df = df[[col for col in essential_cols if col in df.columns]]

print(f"Letterboxd Movies Dataset Ready: {df.shape[0]} films")
print(f"Films with cast: {df['cast'].notna().sum()}")
print(f"Films with themes: {df['themes'].notna().sum()}")
print(f"Available Columns: {df.columns.tolist()}")
display(df[['name', 'genre', 'themes', 'rating']].head())

In [ ]:
import psutil
import os

SAMPLE_SIZE = 16246  # Use all Letterboxd films
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 3)  # Convert to GB

print("="*70)
print("RECOMMENDATION SYSTEM INITIALIZATION - LETTERBOXD DATASET")
print("="*70)
print(f"\nLetterboxd Dataset: {df.shape[0]:,} films")

df_sample = df.copy()  # Use all data
print(f"Dataset Used: {df_sample.shape[0]:,} films")

# 2. Feature Engineering: Combine genre and themes ONLY for recommendations
# Genre has higher weight in similarity determination
df_sample['features'] = (
    df_sample['genre'].fillna('') + " " +  # Main genre
    df_sample['genre'].fillna('') + " " +  # Repeat for higher weight
    df_sample['themes'].fillna('')  # Add themes for content-based similarity
)

# 3. TF-IDF Vectorization
print(f"\nTF-IDF Vectorization Process with Genre and Theme Weighting...")
tfidf = TfidfVectorizer(stop_words='english', max_features=500)
tfidf_matrix = tfidf.fit_transform(df_sample['features'].fillna(''))

# 4. Cosine Similarity Calculation with sparse matrix (memory efficient)
print(f"Computing similarity matrix for {df_sample.shape[0]:,} films...")
cosine_sim = cosine_similarity(tfidf_matrix, dense_output=False)

# 5. Memory tracking
mem_after = process.memory_info().rss / (1024 ** 3)
mem_used = mem_after - mem_before

df = df_sample

print(f"\n{'─'*70}")
print(f"MEMORY STATUS:")
print(f"{'─'*70}")
print(f"Memory Before: {mem_before:.2f} GB")
print(f"Memory After: {mem_after:.2f} GB")
print(f"Memory Used: {mem_used:.2f} GB")
print(f"{'─'*70}")

print(f"\n✓ Recommendation system ready for use.")
print(f"✓ Dataset: Letterboxd Movies ({df_sample.shape[0]:,} films)")
print(f"✓ Features: Genre (2x weight) + Themes")
print(f"✓ Memory Efficient: Using only {mem_used:.2f} GB RAM")

In [ ]:
# ===== RECOMMENDATION FUNCTION =====
def get_recommendations(title, cosine_sim=cosine_sim, data=None):
    """
    Returns 9 films with highest similarity scores based on searched title.
    """
    if data is None:
        data = df
        
    # Search: exact match (case-insensitive)
    matching_films = data[data['name'].str.lower() == title.lower()]
    if len(matching_films) == 0:
        # Fallback: partial match if exact match not found
        partial_matches = data[data['name'].str.contains(title, case=False, na=False)]
        if len(partial_matches) == 0:
            return pd.Series([])
        idx = partial_matches.index[0]
    else:
        idx = matching_films.index[0]

    # Extract similarity scores from sparse matrix
    sim_scores = list(enumerate(cosine_sim[idx].toarray().ravel()))
    
    # Sort by score (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:10]  # Top 9 (excluding the film itself)

    movie_indices = [i[0] for i in sim_scores]
    return data['name'].iloc[movie_indices]

# ===== FILM RECOMMENDATION SEARCH =====
# Modify the variable below with the film title you want to search for

judul_film = 'inception'

print(f"Searching recommendations for: '{judul_film}'")

try:
    rekomendasi = get_recommendations(judul_film)
    if len(rekomendasi) > 0:
        print(f"\nFilms Recommended Based on '{judul_film}':\n")
        for i, film in enumerate(rekomendasi, 1):
            print(f"  {i}. {film}")
    else:
        print(f"\n✗ Film '{judul_film}' not found in dataset.")
        
except Exception as e:
    print(f"\n✗ Error: {e}")

In [ ]:

print(f"DETAILED RECOMMENDATIONS FOR: '{judul_film.upper()}'")


try:
    def get_detailed_recommendations(title, cosine_sim=cosine_sim):
        """
        Display 10 films with detailed information based on similarity score.
        """
        # Film search - exact match first
        matching_films = df[df['name'].str.lower() == title.lower()]
        if len(matching_films) == 0:
            # Try partial match
            partial_matches = df[df['name'].str.contains(title, case=False, na=False)]
            if len(partial_matches) == 0:
                return pd.DataFrame(), False  # Return flag: not found
            idx = partial_matches.index[0]
            found_exact = False  # Partial match only
        else:
            idx = matching_films.index[0]
            found_exact = True  # Exact match
        
        # Similarity calculation
        sim_scores = list(enumerate(cosine_sim[idx].toarray().ravel()))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:21]  # Top 10 (exclude the film itself)
        
        movie_indices = [i[0] for i in sim_scores]
        movie_scores = [i[1] for i in sim_scores]
        
        # Build DataFrame with essential columns only
        result = pd.DataFrame({
            'No': range(1, len(movie_indices) + 1),
            'Film Title': df['name'].iloc[movie_indices].values,
            'Genre': df['genre'].iloc[movie_indices].values,
            'Year': df['date'].iloc[movie_indices].astype('Int64').values,
            'Runtime': df['minute'].iloc[movie_indices].astype('Int64').values,
            'Era': df['movie_era'].iloc[movie_indices].values,
            'Cast': df['cast'].iloc[movie_indices].apply(
                lambda x: x if pd.notna(x) else "N/A"
            ).values,
            'Rating': df['rating'].iloc[movie_indices].apply(
                lambda x: f"{x:.2f}" if pd.notna(x) else "N/A"
            ).values,
            'Description': df['description'].iloc[movie_indices].apply(
                lambda x: x if pd.notna(x) else "N/A"
            ).values,
            'Match Score': [f"{score:.4f}" for score in movie_scores]
        })
        return result, found_exact

    result_df, found_exact = get_detailed_recommendations(judul_film)
    if not result_df.empty:
        if not found_exact:
            print(f" Film '{judul_film}' not found. Showing results for similar title.\n")
        # Display as formatted table
        display(result_df)
        print(f" Recommendations based on genre, description, and film era similarity.")
    else:
        print(f"✗ Film '{judul_film}' not found in dataset.")
        
except Exception as e:
    print(f"✗ Error: {str(e)}")

In [ ]:
import pickle
import scipy.sparse as sp

# Save DataFrame sebagai CSV (safe, portable)
df.to_csv('../processed/movie_list.csv', index=False)

# Save sparse matrix dengan scipy format
sp.save_npz('../processed/similarity_model.npz', cosine_sim)

print(" Files saved successfully!")